In [1]:
from reconciliation import get_sessions

file = '../files/aulas-2026-09-03.xlsx'
sessions, _ = get_sessions(None, file, True)

sessions


,customer,Data,comments,status,minutes,service,status_row,status_type,month
0,Jefferson,2022-10-06,NaN,realizada,60,aula/canto,ok,realizada/reposicao,2022-10
1,Jefferson,2022-09-30,NaN,realizada,60,aula/canto,ok,realizada/reposicao,2022-09
2,Jefferson,2022-09-23,NaN,realizada,30,aula/canto,ok,realizada/reposicao,2022-09
3,Jefferson,2022-09-15,NaN,realizada,30,aula/canto,ok,realizada/reposicao,2022-09
4,Jefferson,2022-09-08,NaN,realizada,60,aula/canto,ok,realizada/reposicao,2022-09
...,...,...,...,...,...,...,...,...,...
2728,NaN,NaT,NaN,ignorar,0,aula/canto,error,ignorar,NaN
2729,rafael,NaT,violao // sound healing (Flip) // light chest,ignorar,0,aula/canto,error,ignorar,NaN
2730,francisco sá - aula 2,NaT,another day old (\nEddie Dalton - Another Day ...,ignorar,0,aula/canto,error,ignorar,NaN
2731,francisco sá - aula 2,2026-05-05,another day old (\nEddie Dalton - Another Day ...,ignorar,0,aula/canto,error,ignorar,2026-05


In [2]:
sessions_final = sessions[sessions['status_row'] == 'ok']


In [8]:

import requests
import pandas as pd
import unicodedata
from requests.exceptions import ConnectionError, Timeout

endpoint = 'http://192.168.1.138:8083/api/session/create'

for index, row in sessions_final.iterrows():
    customer_nickname = (
        unicodedata.normalize("NFKD", str(row["customer"]))
        .encode("ascii", "ignore")
        .decode("ascii")
        .lower()
        .strip()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
    )
    json_data = {
        "nickname": customer_nickname,
        "date": row['Data'].strftime('%Y-%m-%d'),
        "minutes": int(row["minutes"]),
        "status": str(row["status"]),
        "service": str(row["service"]),
        "comments": row["comments"] if pd.notna(row["comments"]) else ""
    }
    try:
        resposta = requests.post(endpoint, json=json_data)
    except ConnectionError as e:
        print(f"Erro: A conexão foi recusada pelo servidor remoto. Detalhes: {e}")
    except Timeout as e:
        print(f"Erro: A requisição excedeu o tempo limite estabelecido. {e}")
    except requests.exceptions.RequestException as e:
        print(f"Ocorreu um erro genérico no requests: {e}")
    else:
        try:
            resposta_json = resposta.json()
        except ValueError:
            resposta_json = resposta.text

        print(
            f"Status Code: {resposta.status_code}, "
            f"Response JSON: {resposta_json}, "
            f"Data: {json_data}"
        )

Status Code: 200, Response JSON: {'http_code': 200, 'status': 'success', 'message': 'session created', 'session_id': 3}, Data: {'nickname': 'jefferson', 'date': '2022-10-06', 'minutes': 60, 'status': 'realizada', 'service': 'aula/canto', 'comments': ''}
Status Code: 200, Response JSON: {'http_code': 200, 'status': 'success', 'message': 'session created', 'session_id': 4}, Data: {'nickname': 'jefferson', 'date': '2022-09-30', 'minutes': 60, 'status': 'realizada', 'service': 'aula/canto', 'comments': ''}
Status Code: 200, Response JSON: {'http_code': 200, 'status': 'success', 'message': 'session created', 'session_id': 5}, Data: {'nickname': 'jefferson', 'date': '2022-09-23', 'minutes': 30, 'status': 'realizada', 'service': 'aula/canto', 'comments': ''}
Status Code: 200, Response JSON: {'http_code': 200, 'status': 'success', 'message': 'session created', 'session_id': 6}, Data: {'nickname': 'jefferson', 'date': '2022-09-15', 'minutes': 30, 'status': 'realizada', 'service': 'aula/canto', 

In [7]:
sessions_final.count()

customer       2715
Data           2715
comments       1355
status         2715
minutes        2715
service        2715
status_row     2715
status_type    2715
month          2715
dtype: int64